# HW1 starter notebook

CS 460200 Introduction to Machine Learning, Fall 2026.

This notebook does three things and nothing more: it loads `tracks.csv`, shows the
level of evidence expected for an audit finding in Section 3 of the handout, and gives
you a place to collect your cleaning steps. The analysis itself is yours.

Data: *Spotify Tracks Dataset*, distributed on Hugging Face as
`maharshipandya/spotify-tracks-dataset`. The course copy has the row index column removed
and is otherwise unchanged. Cite the source in your report.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 30)
RANDOM_STATE = 0  # use this everywhere, so your results are reproducible

DATA_URL = 'https://github.com/George930502/IntroML-f26/releases/download/data/tracks.csv'
LOCAL_PATH = None  # set this if you downloaded the file from eeclass instead

df = pd.read_csv(LOCAL_PATH or DATA_URL)
df.shape

ModuleNotFoundError: No module named 'numpy'

## First look

Before anything else, find out what one row is, what the columns hold, and which of them
are numbers, categories, or identifiers wearing a number's clothing.

In [ ]:
display(df.head(3))
df.describe().T[['mean', 'std', 'min', '50%', 'max']].round(3)

## What an audit finding looks like

Section 3 of the handout asks for at most eight findings, ordered by how much each would
distort a model trained on this file. Each finding needs four parts:

| Part | What it means |
|:--|:--|
| Evidence | one number or one figure that establishes the fact |
| Mechanism | why you believe the data ended up this way |
| Decision | what you do about it |
| Cost | what your decision throws away or risks |

Below is one worked finding in that format. It is deliberately a small one: it affects
0.14% of the rows. Part of your task is to find the findings that matter more than this
one, and to argue for the order you put them in.

In [ ]:
# Evidence
print(df.time_signature.value_counts().sort_index(), '\n')

zero_ts = df.time_signature == 0
print(f'rows with time_signature == 0: {zero_ts.sum()} ({zero_ts.mean():.2%})')
print('of those, tempo == 0        :', (df.loc[zero_ts, 'tempo'] == 0).sum())
print('of those, danceability == 0 :', (df.loc[zero_ts, 'danceability'] == 0).sum())
print('\nwhere they sit:')
print(df.loc[zero_ts, 'track_genre'].value_counts().head(4))
print('\nmedian duration (s): flagged', round(df.loc[zero_ts, 'duration_ms'].median() / 1000),
      '| all rows', round(df.duration_ms.median() / 1000))

### Finding (example): a block of rows carries zeros instead of measurements

**Evidence.** 163 rows (0.14%) have `time_signature == 0`, which is not a meter. In 157 of
them `tempo` and `danceability` are also exactly 0. 138 of the 163 are in the `sleep`
genre, and their median length is 93 s against 213 s for the file as a whole.

**Mechanism.** These are not measurements. The audio analyzer failed on these tracks, most
of them short ambient pieces with no steady beat, and wrote zeros where it had nothing to
report.

**Decision.** Treat the affected values as missing rather than as zeros. I flag the rows
and exclude them from any analysis that uses `tempo`, `danceability`, or `time_signature`.

**Cost.** The loss is 0.14% of the file overall, but 13.8% of the `sleep` genre, so any
genre-level statistic for `sleep` is now computed on 862 tracks and is biased toward the
tracks the analyzer could handle. I say so wherever `sleep` appears in a result.

In [ ]:
def clean_tracks(df):
    """Return the table your analysis uses.

    Every decision from your audit belongs here, one block per finding, with a comment
    naming the finding it implements. Section 7 asks what your cleaning keeps and what it
    discards, so keep it readable.
    """
    out = df.copy()

    # Finding: failed audio analysis stored as zeros (worked example above).
    failed = out.time_signature == 0
    out.loc[failed, ['tempo', 'danceability', 'time_signature']] = np.nan

    # TODO: your findings go here.

    return out


clean = clean_tracks(df)
print(df.shape, '->', clean.shape)
clean[['tempo', 'danceability', 'time_signature']].isna().sum()

## Your turn

The headings below match the handout. Keep the report and the notebook in the same order,
so a reader can move between them.

Two practical notes. Set `random_state=RANDOM_STATE` on anything that samples or
initializes randomly, or your numbers will move between runs and you will not be able to
defend them at the demo. The report allows at most 8 figures, so decide which results
deserve one.

## 3. Auditing the file

## 4. The geometry of the feature space

## 5. Structure in the catalog

## 6. Two regression targets

## 7. Conclusions